# Phase 13 — Repeated runs for all remaining experiments (seeds 1 and 2)
So that every result can be reported as mean +- standard deviation over three seeds (42, 1, 2):
* Configurations 2 and 3: SkyShield-IDS, SkyShield-IDS-S (distilled from the seed-42 teacher of the same configuration)
  and the student without distillation, seeds 1 and 2 (12 runs)
* Cross-dataset models in both directions, seeds 1 and 2 (4 runs)
* Optional: the softened-weight teacher in the merged configuration, seeds 1 and 2 (2 runs; RUN_SQRT_SEEDS)
* INT8 post-training quantization (variant A) of every repeated teacher and student, including the merged seeds of Phase 11
* Random Forest baselines, seeds 1 and 2, in all three configurations (XGBoost is deterministic)
Settings as in Phases 5, 7 and 8. Progress logs and epoch checkpoints as before; re-run Cell 2 after any interruption.
Outputs: outputs_v3/seeds_v2/ (results/, checkpoints/, models/, int8/, rf/, seeds_v2_log.txt, summary.json)

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — settings, models, helpers
import os, json, time, gc, math, platform
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

FEAT    = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/'
TEACHER = '/content/drive/MyDrive/Thesis/outputs_v3/final_v1/'
OUT     = '/content/drive/MyDrive/Thesis/outputs_v3/seeds_v2/'
KDDIR   = '/content/drive/MyDrive/Thesis/outputs_v3/kd_v1/'
SEEDS1  = '/content/drive/MyDrive/Thesis/outputs_v3/seeds_v1/'
for sub in ('results/', 'checkpoints/', 'models/'):
    os.makedirs(OUT + sub, exist_ok=True)
SEED, LR, BATCH, EPOCHS, PATIENCE = 42, 1e-4, 128, 100, 5
T_KD, ALPHA, WIDTH_DIV = 3.0, 0.4, 4
N_CLASSES = 8
CLASS_NAMES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
SRC = {'cicids': 'CICIDS2017', 'iot': 'CIC-IoT2023'}
RUN_SQRT_SEEDS = True   # also repeat the softened-weight teacher (merged) with seeds 1 and 2; set False to skip
# (name, kind, data, seed, class-weight mode); kind: teacher | kd | nokd | cross
RUNS = []
for cfg in ('config2_cicids2017', 'config3_ciciot2023'):           # Configuration 2 first (smallest), then 3
    RUNS += [(f'teacher_{cfg}_seed{s}', 'teacher', cfg, s, 'standard') for s in (1, 2)]
    RUNS += [(f'{k}_{cfg}_seed{s}', k, cfg, s, 'standard') for k in ('kd', 'nokd') for s in (1, 2)]
for a_, b_ in ((SRC['cicids'], SRC['iot']), (SRC['iot'], SRC['cicids'])):
    RUNS += [(f'cross_{a_}_to_{b_}_seed{s}', 'cross', (a_, b_), s, 'standard') for s in (1, 2)]
if RUN_SQRT_SEEDS:
    RUNS += [(f'teacher_sqrtweights_config1_merged_seed{s}', 'teacher', 'config1_merged', s, 'sqrt') for s in (1, 2)]
TOTAL_RUNS = len(RUNS)
LOGFILE = OUT + 'seeds_v2_log.txt'

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + ' ' + msg
    print(line, flush=True)
    with open(LOGFILE, 'a') as f:
        f.write(line + '\n')

def hm(minutes):
    minutes = max(0.0, float(minutes))
    return f'{int(minutes // 60)} h {int(round(minutes % 60)):02d} min' if minutes >= 60 else f'{minutes:.1f} min'

# ---------- models and training helpers (as in Phases 5 and 7)
def eca(x):
    c = x.shape[-1]
    w = layers.GlobalAveragePooling1D()(x)
    w = layers.Reshape((c, 1))(w)
    w = layers.Conv1D(1, 3, padding='same', use_bias=False, activation='sigmoid')(w)
    w = layers.Reshape((1, c))(w)
    return layers.Multiply()([x, w])

def build(variant, n_features, n_classes=N_CLASSES):
    """A: CNN+GAP | B: +ECA | C: +skip | D: +ECA +skip | E: +2xECA +skip (identical to the ablation study)"""
    inp = keras.Input((n_features, 1))
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    if variant == 'E':
        x = eca(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    if variant in ('B', 'D', 'E'):
        x = eca(x)
    z = layers.GlobalAveragePooling1D()(x)
    if variant in ('C', 'D', 'E'):
        s = layers.Dense(128, activation='relu')(layers.Flatten()(inp))
        z = layers.Concatenate()([z, s])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(n_classes, activation='softmax')(z)
    return keras.Model(inp, out, name=f'Model_{variant}')

def counts(y):
    c = np.bincount(y, minlength=N_CLASSES)
    return {CLASS_NAMES[i]: int(n) for i, n in enumerate(c) if n}

def class_weights(y):  # noqa
    """w_c = N / (K * N_c) over the K classes present in the training data (Kamal & Mashaly, 2025)."""
    c = np.bincount(y, minlength=N_CLASSES); present = np.flatnonzero(c)
    return {int(k): float(len(y) / (len(present) * c[k])) for k in present}

def result_path(name):
    return OUT + f'results/{name}.json'

def ckpt_dir(name):
    return OUT + f'checkpoints/{name}/'

def n_done():
    return sum(os.path.exists(result_path(r[0])) for r in RUNS)

def save_atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def val_metrics(model, Xv, yv):
    """Loss, accuracy, Macro F1 and per-class F1 on the complete validation split (one prediction pass)."""
    p = model.predict(Xv, batch_size=8192, verbose=0)
    loss = float(-np.mean(np.log(np.clip(p[np.arange(len(yv)), yv], 1e-7, 1.0))))   # unweighted cross-entropy
    yp = p.argmax(1); labels = np.unique(yv)
    f1c = f1_score(yv, yp, average=None, labels=labels, zero_division=0)
    return loss, float(accuracy_score(yv, yp)), float(f1c.mean()), {CLASS_NAMES[k]: round(float(v), 3) for k, v in zip(labels, f1c)}

def evaluate(model, X, y, labels=None):
    """Metrics on a test split; labels = the classes over which Macro F1 is averaged (default: those present in y)."""
    t0 = time.time()
    pred = model.predict(X, batch_size=8192, verbose=0).argmax(1)
    sec = time.time() - t0
    labels = np.unique(y) if labels is None else np.asarray(labels)
    pr, rc, f1c, sup = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    return dict(macro_f1=float(f1c.mean()), accuracy=float(accuracy_score(y, pred)),
                weighted_f1=float(f1_score(y, pred, average='weighted', labels=labels, zero_division=0)),
                per_class={CLASS_NAMES[k]: dict(precision=float(p_), recall=float(r_), f1=float(f_), support=int(s_))
                           for k, p_, r_, f_, s_ in zip(labels, pr, rc, f1c, sup)},
                confusion_matrix=confusion_matrix(y, pred, labels=labels).tolist(),
                labels=[CLASS_NAMES[k] for k in labels], test_rows=int(len(y)),
                predicted_outside_labels=int((~np.isin(pred, labels)).sum()),
                inference_seconds=round(sec, 2), inference_ms_per_1000=round(1000 * 1000 * sec / max(1, len(y)), 3))

def log_eval(prefix, r):
    log(f"{prefix}: Macro F1 {r['macro_f1']:.4f} | accuracy {r['accuracy']:.4f} | weighted F1 {r['weighted_f1']:.4f} | "
        f"{r['test_rows']:,} test rows | inference {r['inference_ms_per_1000']:.3f} ms per 1,000 records")
    log('   per class (precision / recall / F1 / support): ' + '; '.join(
        f"{k} {v['precision']:.3f}/{v['recall']:.3f}/{v['f1']:.3f}/{v['support']:,}" for k, v in r['per_class'].items()))
    if r['predicted_outside_labels']:
        log(f"   {r['predicted_outside_labels']:,} records were predicted as a class outside the evaluated classes (counted as errors)")

# ---------- data

def eca(x):
    c = x.shape[-1]
    w = layers.GlobalAveragePooling1D()(x)
    w = layers.Reshape((c, 1))(w)
    w = layers.Conv1D(1, 3, padding='same', use_bias=False, activation='sigmoid')(w)
    w = layers.Reshape((1, c))(w)
    return layers.Multiply()([x, w])

def build_student(n_features, div=WIDTH_DIV, n_classes=N_CLASSES):
    """Model E (Conv 64 -> ECA -> Conv 128 -> ECA -> GAP, skip Dense 128, Dense 64) with every width divided by div."""
    inp = keras.Input((n_features, 1))
    x = layers.Conv1D(64 // div, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = eca(x)
    x = layers.Conv1D(128 // div, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = eca(x)
    z = layers.GlobalAveragePooling1D()(x)
    s = layers.Dense(128 // div, activation='relu')(layers.Flatten()(inp))
    z = layers.Concatenate()([z, s])
    z = layers.Dense(64 // div, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(n_classes, activation='softmax')(z)
    return keras.Model(inp, out, name=f'Student_E_div{div}')

def _kd(y_true, y_pred, alpha):
    """y_true = [label, teacher probabilities (8)]; y_pred = student probabilities."""
    lab = keras.ops.cast(y_true[:, 0], 'int32'); pt = y_true[:, 1:]
    ce = keras.losses.sparse_categorical_crossentropy(lab, y_pred)
    if alpha == 0:
        return ce
    ls = keras.ops.log(keras.ops.clip(y_pred, 1e-7, 1.0)); lt = keras.ops.log(keras.ops.clip(pt, 1e-7, 1.0))
    ps, pt_t = keras.ops.softmax(ls / T_KD, axis=-1), keras.ops.softmax(lt / T_KD, axis=-1)
    kl = keras.ops.sum(pt_t * (keras.ops.log(keras.ops.clip(pt_t, 1e-7, 1.0)) - keras.ops.log(keras.ops.clip(ps, 1e-7, 1.0))), axis=-1)
    return alpha * T_KD ** 2 * kl + (1 - alpha) * ce

@tf.keras.utils.register_keras_serializable(package='kd')
def kd_loss(y_true, y_pred):
    return _kd(y_true, y_pred, ALPHA)

@tf.keras.utils.register_keras_serializable(package='kd')
def ce_only_loss(y_true, y_pred):
    return _kd(y_true, y_pred, 0.0)

def load_config(cfg, split):
    X = np.load(FEAT + cfg + f'/X_{split}.npy').astype(np.float32, copy=False)
    y = np.load(FEAT + cfg + f'/y_{split}.npy').astype(np.int64)
    return X, y

class ProgressCheckpoint(keras.callbacks.Callback):
    """Detailed progress (every 10% of an epoch, with time left), complete-validation evaluation, early stopping
    (patience 5, best weights kept) and an atomic checkpoint after every epoch."""
    def __init__(self, tag, d, Xv, yv, steps, state, monitor='loss'):
        super().__init__()
        self.tag, self.d, self.Xv, self.yv, self.steps, self.state = tag, d, Xv, yv, steps, state
        self.monitor = monitor   # 'loss': unweighted validation cross-entropy (as in Phases 4-7); 'f1': validation Macro F1
        self.marks = {max(1, int(steps * q / 10)) for q in range(1, 10)}
        self.t_run = time.time() - state['minutes'] * 60
    def on_epoch_begin(self, epoch, logs=None):
        self.t_ep, self.cur = time.time(), epoch + 1
        log(f"{self.tag} | epoch {self.cur} started ({self.steps:,} steps of {BATCH})")
    def on_train_batch_end(self, batch, logs=None):
        if batch + 1 in self.marks:
            el = (time.time() - self.t_ep) / 60; frac = (batch + 1) / self.steps
            log(f"{self.tag} | epoch {self.cur} | step {batch + 1:,}/{self.steps:,} ({100 * frac:.0f}%) | "
                f"running loss {logs['loss']:.4f} | {hm(el)} elapsed, about {hm(el / frac - el)} left in this epoch")
    def on_epoch_end(self, epoch, logs=None):
        st, d = self.state, self.d
        t_train = (time.time() - self.t_ep) / 60
        vloss, vacc, vf1, vpc = val_metrics(self.model, self.Xv, self.yv)
        score = vloss if self.monitor == 'loss' else -vf1
        improved = score < st['best']
        if improved:
            self.model.save_weights(d + 'best.tmp.weights.h5'); os.replace(d + 'best.tmp.weights.h5', d + 'best.weights.h5')
            st.update(best=score, best_ep=epoch + 1, wait=0)
        else:
            st['wait'] += 1
        st.update(epoch=epoch + 1, minutes=(time.time() - self.t_run) / 60)
        st.setdefault('history', []).append(dict(epoch=epoch + 1, loss=round(float(logs['loss']), 5), val_loss=round(vloss, 5),
                                                 val_acc=round(vacc, 5), val_macro_f1=round(vf1, 5)))
        self.model.save(d + 'last.tmp.keras'); os.replace(d + 'last.tmp.keras', d + 'last.keras')
        save_atomic_json(st, d + 'state.json')
        ep_min = st['minutes'] / st['epoch']
        log(f"{self.tag} | epoch {epoch + 1} done | train {t_train:.1f} min + validation {(time.time() - self.t_ep) / 60 - t_train:.1f} min | "
            f"loss {logs['loss']:.4f} | val_loss {vloss:.4f}{' (best)' if improved else ''} | val_acc {vacc:.4f} | val_macroF1 {vf1:.4f}")
        log(f"{self.tag} |   val F1 per class: {vpc}")
        log(f"{self.tag} |   best epoch {st['best_ep']} | no improvement {st['wait']}/{PATIENCE} | run time {hm(st['minutes'])} | "
            f"at most {PATIENCE - st['wait']} more epochs without improvement (about {hm((PATIENCE - st['wait']) * ep_min)}) | checkpoint saved")
        if st['wait'] >= PATIENCE:
            log(f"{self.tag} | early stopping: no improvement for {PATIENCE} epochs; best epoch {st['best_ep']} will be restored")
            self.model.stop_training = True


def class_weights_mode(y, mode):
    cw = class_weights(y)
    return {k: (math.sqrt(v) if mode == 'sqrt' else v) for k, v in cw.items()}

def fpr_far(r):
    """Add per-class FPR and the false alarm rate (benign records classified as any attack) to an evaluate() result."""
    cm = np.array(r['confusion_matrix']); tot = cm.sum(); labs = r['labels']
    for i, k in enumerate(labs):
        fp = cm[:, i].sum() - cm[i, i]; tn = tot - cm[i].sum() - fp
        r['per_class'][k]['fpr'] = float(fp / (fp + tn)) if fp + tn else 0.0
    if 'BENIGN' in labs:
        b = labs.index('BENIGN'); r['false_alarm_rate'] = float(1 - cm[b, b] / cm[b].sum())
    return r


# ---------- cross-dataset data (as in Phase 5)
def merged_raw(split, source):
    """Rows of one dataset in the merged configuration, with the merged Min–Max scaling (and log transform) undone."""
    sp = json.load(open(FEAT + 'config1_merged/spec.json'))
    X, y = load_config('config1_merged', split)
    s = np.load(FEAT + f'config1_merged/source_{split}.npy')
    keep = s == source
    X, y = X[keep].astype(np.float64), y[keep]
    mn, mx = np.array(sp['train_min']), np.array(sp['train_max'])
    use_log = np.array([f in sp['log_features'] for f in sp['features']])
    lo = np.where(use_log, np.log1p(np.maximum(mn, 0)), mn)
    hi = np.where(use_log, np.log1p(np.maximum(mx, 0)), mx)
    rng = np.where(hi > lo, hi - lo, 1.0)
    X = X * rng + lo
    X[:, use_log] = np.expm1(X[:, use_log])
    return X, y, sp

def fit_scaling(train, feats, wide_range):
    """Same rule as in Section 3.6.3 (log(1+x) for wide-range non-negative features, then Min–Max), fitted on train only."""
    mn, mx = train.min(0), train.max(0)
    use_log = ((mx - mn) > wide_range) & (mn >= 0)
    lo = np.where(use_log, np.log1p(np.maximum(mn, 0)), mn)
    hi = np.where(use_log, np.log1p(np.maximum(mx, 0)), mx)
    rng = np.where(hi > lo, hi - lo, 1.0)
    def tf_(x):
        x = np.array(x, copy=True)
        x[:, use_log] = np.log1p(np.maximum(x[:, use_log], 0))
        return ((x - lo) / rng).astype(np.float32)
    return tf_, [f for f, u in zip(feats, use_log) if u]

def get_data(name, kind, a, b):
    if kind == 'in':
        Xtr, ytr = load_config(a, 'train'); Xv, yv = load_config(a, 'val'); Xte, yte = load_config(a, 'test')
        extra = {}
        if a == 'config1_merged':                         # test rows of each dataset, for the per-dataset breakdown
            extra['source_test'] = np.load(FEAT + a + '/source_test.npy')
        return Xtr, ytr, Xv, yv, {'own': (Xte, yte)}, None, extra
    # cross-dataset: train/val of dataset a, test of dataset b (and of a, as in-domain baseline)
    Rtr, ytr, sp = merged_raw('train', a); Rv, yv, _ = merged_raw('val', a)
    Rte_a, yte_a, _ = merged_raw('test', a); Rte_b, yte_b, _ = merged_raw('test', b)
    shared = sorted(set(np.unique(ytr)) & set(np.unique(yte_b)))
    dropped = sorted((set(np.unique(ytr)) | set(np.unique(yte_b))) - set(shared))
    log(f'{name}: classes in both datasets {[CLASS_NAMES[k] for k in shared]} | left out (present in only one): {[CLASS_NAMES[k] for k in dropped]}')
    def keep(R, y):
        m = np.isin(y, shared); return R[m], y[m]
    Rtr, ytr = keep(Rtr, ytr); Rv, yv = keep(Rv, yv); Rte_a, yte_a = keep(Rte_a, yte_a); Rte_b, yte_b = keep(Rte_b, yte_b)
    tf_, log_feats = fit_scaling(Rtr, sp['features'], sp['wide_range_threshold'])
    Xtr, Xv, Xa, Xb = tf_(Rtr), tf_(Rv), tf_(Rte_a), tf_(Rte_b)
    out_rng = float(((Xb < 0) | (Xb > 1)).any(1).mean())
    log(f'{name}: scaling refitted on the {a} training rows only | log(1+x) features: {log_feats}')
    log(f'{name}: {100 * out_rng:.1f}% of the {b} test rows have at least one feature outside the training range [0, 1] '
        f'(kept as they are, no clipping)')
    del Rtr, Rv, Rte_a, Rte_b; gc.collect()
    return Xtr, ytr, Xv, yv, {'in_domain': (Xa, yte_a), 'cross': (Xb, yte_b)}, shared, dict(scaling_log_features=log_feats,
                                                                                         target_rows_outside_train_range=out_rng)


log('=' * 100)
log(f'Repeated runs v2 (all configurations) | TensorFlow {tf.__version__} | Python {platform.python_version()} | GPU: '
    f'{[g.name for g in tf.config.list_physical_devices("GPU")] or "none"}')
log(f'{n_done()} of {TOTAL_RUNS} runs already finished')

In [ ]:
# %% Cell 2 — train and test all runs (resumable: re-run this cell after any interruption)
T0 = time.time(); cache = {}
for i, (name, kind, data, seed, wmode) in enumerate(RUNS, 1):
    if os.path.exists(result_path(name)):
        log(f'run {i} of {TOTAL_RUNS} ({name}): already finished, skipping'); continue
    key = data if kind != 'cross' else ('cross',) + tuple(data)
    if cache.get('key') != key:                                   # load the data of this configuration once
        cache.clear(); gc.collect()
        if kind == 'cross':
            Xtr, ytr, Xv, yv, tests, shared, extra = get_data(name, 'cross', *data)
        else:
            Xtr, ytr = load_config(data, 'train'); Xv, yv = load_config(data, 'val'); Xte, yte = load_config(data, 'test')
            tests, shared, extra = {'own': (Xte, yte)}, None, {}
        cache = dict(key=key, Xtr=Xtr[..., None], ytr=ytr, Xv=Xv[..., None], yv=yv, tests=tests, shared=shared, extra=extra, ypack=None)
        del Xtr, Xv; gc.collect()
    c = cache; Xtr, ytr, Xv, yv = c['Xtr'], c['ytr'], c['Xv'], c['yv']
    if kind in ('kd', 'nokd') and c['ypack'] is None:             # soft targets of the seed-42 teacher of this configuration
        teacher = keras.models.load_model(TEACHER + f'models/final_{data}.keras')
        pt = teacher.predict(Xtr, batch_size=8192, verbose=0).astype(np.float32)
        c['ypack'] = np.concatenate([ytr[:, None].astype(np.float32), pt], 1)
        log(f'{data}: teacher (seed 42) probabilities computed for {len(ytr):,} training rows'); del teacher, pt; gc.collect()
    cw = class_weights_mode(ytr, wmode)
    steps = int(math.ceil(len(ytr) / BATCH))
    log(f'--- run {i} of {TOTAL_RUNS}: {name} | {kind} | seed {seed} | class weights {wmode} | {len(ytr):,} training rows, {Xtr.shape[1]} features ---')
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    d = ckpt_dir(name); os.makedirs(d, exist_ok=True)
    if kind in ('teacher', 'cross'):
        fit_kw = dict(x=Xtr, y=ytr, class_weight=cw)
        make = lambda: build('E', Xtr.shape[1]); loss = 'sparse_categorical_crossentropy'
    else:
        wvec = np.zeros(N_CLASSES, np.float32)
        for k, v in cw.items(): wvec[k] = v
        fit_kw = dict(x=Xtr, y=c['ypack'], sample_weight=wvec[ytr])
        make = lambda: build_student(Xtr.shape[1]); loss = kd_loss if kind == 'kd' else ce_only_loss
    if os.path.exists(d + 'state.json') and os.path.exists(d + 'last.keras'):
        state = json.load(open(d + 'state.json'))
        model = keras.models.load_model(d + 'last.keras')
        log(f"{name} --- RESUMING after epoch {state['epoch']} (best epoch {state['best_ep']}, no improvement {state['wait']}/{PATIENCE}) ---")
    else:
        state = dict(epoch=0, best=float('inf'), best_ep=0, wait=0, minutes=0.0, history=[])
        model = make(); model.compile(optimizer=keras.optimizers.Adam(LR), loss=loss)
    params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    log(f'{name} | {params:,} trainable parameters | {steps:,} steps per epoch')
    if not (state['wait'] >= PATIENCE or state['epoch'] >= EPOCHS):
        model.fit(batch_size=BATCH, epochs=EPOCHS, initial_epoch=state['epoch'], verbose=0, shuffle=True,
                  callbacks=[ProgressCheckpoint(name, d, Xv, yv, steps, state, monitor='loss')], **fit_kw)
    model.load_weights(d + 'best.weights.h5')
    model.save(OUT + f'models/{name}.tmp.keras'); os.replace(OUT + f'models/{name}.tmp.keras', OUT + f'models/{name}.keras')
    res = dict(run=name, kind=kind, data=data if kind != 'cross' else list(data), seed=seed, class_weights_mode=wmode,
               epochs=state['epoch'], best_epoch=state['best_ep'], train_minutes=round(state['minutes'], 2), params=params,
               history=state.get('history', []), tests={})
    for k, (Xt, yt) in c['tests'].items():
        r = evaluate(model, Xt[..., None], yt, labels=c['shared'])
        if 'BENIGN' in r['labels']: r = fpr_far(r)
        res['tests'][k] = r; log_eval(f'{name} RESULT [{k} test]', r)
        if 'false_alarm_rate' in r: log(f"   false alarm rate {r['false_alarm_rate'] * 100:.2f}%")
    if kind == 'cross':
        ind, crs = res['tests']['in_domain']['macro_f1'], res['tests']['cross']['macro_f1']
        res['macro_f1_drop_percent'] = round(100 * (ind - crs) / ind, 2) if ind > 0 else None
        log(f"{name}: in-domain Macro F1 {ind:.4f} -> cross-dataset Macro F1 {crs:.4f} | drop {res['macro_f1_drop_percent']}%")
    save_atomic_json(res, result_path(name))
    log(f"{name}: {res['epochs']} epochs (best {res['best_epoch']}) | {hm(res['train_minutes'])} | "
        f"finished {n_done()} of {TOTAL_RUNS} runs | session {hm((time.time() - T0) / 60)}")
    del model; gc.collect()
log(f'repeated runs v2: {n_done()} of {TOTAL_RUNS} runs finished')

In [ ]:
# %% Cell 3 — INT8 post-training quantization of every repeated model (variant A: 500 training records stratified by class)
from sklearn.model_selection import train_test_split
try:
    Interpreter = tf.lite.Interpreter
except AttributeError:
    os.system('pip -q install ai-edge-litert'); from ai_edge_litert.interpreter import Interpreter
os.makedirs(OUT + 'int8/', exist_ok=True)
def tflite_predict(path, X, bs=8192):
    it = Interpreter(model_path=path); inp, out = it.get_input_details()[0], it.get_output_details()[0]
    scale, zp = inp['quantization']; preds = []
    for s in range(0, len(X), bs):
        xb = X[s:s + bs]
        if inp['dtype'] == np.int8: xb = np.clip(np.round(xb / scale + zp), -128, 127).astype(np.int8)
        it.resize_tensor_input(inp['index'], xb.shape); it.allocate_tensors(); it.set_tensor(inp['index'], xb); it.invoke()
        preds.append(it.get_tensor(out['index']).argmax(1))
    return np.concatenate(preds)
Q = []   # (name, configuration, model path)
for cfg in ('config2_cicids2017', 'config3_ciciot2023'):
    Q += [(f'{k}_{cfg}_seed{s}', cfg, OUT + f'models/{k}_{cfg}_seed{s}.keras') for k in ('teacher', 'kd') for s in (1, 2)]
Q += [(f'{k}_config1_merged_seed{s}', 'config1_merged', SEEDS1 + f'models/{k}_seed{s}.keras') for k in ('teacher', 'kd') for s in (1, 2)]
qc = {}
for name, cfg, path in Q:
    rp = OUT + f'int8/{name}.json'
    if os.path.exists(rp) or not os.path.exists(path):
        log(f'INT8 {name}: ' + ('already done' if os.path.exists(rp) else 'model not found yet, skipped')); continue
    if qc.get('cfg') != cfg:
        qc.clear(); gc.collect()
        Xtr_ = np.load(FEAT + cfg + '/X_train.npy', mmap_mode='r'); ytr_ = np.load(FEAT + cfg + '/y_train.npy')
        idx, _ = train_test_split(np.arange(len(ytr_)), train_size=500, stratify=ytr_, random_state=42)
        qc = dict(cfg=cfg, rep=np.asarray(Xtr_[np.sort(idx)], np.float32)[..., None],
                  Xte=np.load(FEAT + cfg + '/X_test.npy').astype(np.float32)[..., None], yte=np.load(FEAT + cfg + '/y_test.npy').astype(np.int64))
        del Xtr_
    model = keras.models.load_model(path, compile=False)
    pf = model.predict(qc['Xte'], batch_size=8192, verbose=0).argmax(1)
    conv = tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = lambda: ([qc['rep'][i:i + 1]] for i in range(len(qc['rep'])))
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    b8 = conv.convert(); mp = OUT + f'int8/{name}_int8.tflite'; open(mp, 'wb').write(b8)
    p8 = tflite_predict(mp, qc['Xte']); labels = np.unique(qc['yte'])
    f32 = float(f1_score(qc['yte'], pf, average='macro', labels=labels, zero_division=0))
    i8 = float(f1_score(qc['yte'], p8, average='macro', labels=labels, zero_division=0))
    save_atomic_json(dict(model=name, config=cfg, macro_f1_float32=f32, macro_f1_int8=i8, change=i8 - f32,
                          agreement=float((p8 == pf).mean()), int8_kb=round(len(b8) / 1024, 1)), rp)
    log(f'INT8 {name}: Macro F1 float32 {f32:.4f} -> INT8 {i8:.4f} ({i8 - f32:+.4f}) | agreement {100 * (p8 == pf).mean():.1f}% | {len(b8) / 1024:.1f} KB')
    del model; gc.collect()
log('INT8 of repeated models: done')

In [ ]:
# %% Cell 4 — Random Forest baselines with seeds 1 and 2 (XGBoost needs no repetition: without row or column
# subsampling, its 'hist' training is deterministic, so a different seed gives the same model)
from sklearn.ensemble import RandomForestClassifier
os.makedirs(OUT + 'rf/', exist_ok=True)
for cfg in ('config1_merged', 'config2_cicids2017', 'config3_ciciot2023'):
    for s in (1, 2):
        name = f'random_forest_{cfg}_seed{s}'; rp = OUT + f'rf/{name}.json'
        if os.path.exists(rp): log(f'{name}: already done'); continue
        Xtr_, ytr_ = load_config(cfg, 'train'); Xte_, yte_ = load_config(cfg, 'test')
        cw = class_weights(ytr_)
        rf = RandomForestClassifier(n_estimators=100, max_samples=min(500_000, len(ytr_)), min_samples_leaf=1,
                                    class_weight=cw, n_jobs=-1, random_state=s)
        t1 = time.time(); rf.fit(Xtr_, ytr_); pred = rf.predict(Xte_)
        labels = np.unique(yte_); f1c = f1_score(yte_, pred, average=None, labels=labels, zero_division=0)
        cm = confusion_matrix(yte_, pred, labels=labels); b = list(labels).index(0)
        res = dict(run=name, seed=s, macro_f1=float(f1c.mean()), accuracy=float(accuracy_score(yte_, pred)),
                   false_alarm_rate=float(1 - cm[b, b] / cm[b].sum()), train_minutes=round((time.time() - t1) / 60, 2))
        save_atomic_json(res, rp)
        log(f"{name}: Macro F1 {res['macro_f1']:.4f} | false alarm {100 * res['false_alarm_rate']:.2f}% | {res['train_minutes']} min")
        del rf, Xtr_, ytr_, Xte_, yte_; gc.collect()
log('Random Forest seeds: done')

In [ ]:
# %% Cell 5 — summary: mean +- SD over three seeds (seed 42 from Phases 5, 7, 8 and 10; seeds 1 and 2 from Phases 11 and 13)
def own(p, test='own'):
    r = json.load(open(p)); t = r['tests'][test]; return t['macro_f1']
F, K, B = TEACHER + 'results/', KDDIR + 'results/', '/content/drive/MyDrive/Thesis/outputs_v3/baselines_v1/results/'
rows = []
for cfg in ('config1_merged', 'config2_cicids2017', 'config3_ciciot2023'):
    for kind, p42, pat in (('teacher', F + f'final_{cfg}.json', 'teacher'), ('kd', K + f'kd_{cfg}.json', 'kd'), ('nokd', K + f'nokd_{cfg}.json', 'nokd')):
        ps = [p42] + ([SEEDS1 + f'results/{pat}_seed{s}.json' for s in (1, 2)] if cfg == 'config1_merged' else [result_path(f'{pat}_{cfg}_seed{s}') for s in (1, 2)])
        v = [own(p) for p in ps if os.path.exists(p)]
        rows.append(dict(config=cfg, model=kind, runs=len(v), values=list(np.round(v, 4)), mean=np.mean(v), sd=np.std(v, ddof=1) if len(v) > 1 else np.nan))
    rv = [json.load(open(B + f'random_forest_{cfg}.json'))['test']['macro_f1']] + [json.load(open(OUT + f'rf/random_forest_{cfg}_seed{s}.json'))['macro_f1'] for s in (1, 2) if os.path.exists(OUT + f'rf/random_forest_{cfg}_seed{s}.json')]
    rows.append(dict(config=cfg, model='random_forest', runs=len(rv), values=list(np.round(rv, 4)), mean=np.mean(rv), sd=np.std(rv, ddof=1) if len(rv) > 1 else np.nan))
for a_, b_ in ((SRC['cicids'], SRC['iot']), (SRC['iot'], SRC['cicids'])):
    ps = [F + f'cross_{a_}_to_{b_}.json'] + [result_path(f'cross_{a_}_to_{b_}_seed{s}') for s in (1, 2)]
    v = [own(p, 'cross') for p in ps if os.path.exists(p)]
    rows.append(dict(config=f'cross {a_} -> {b_}', model='teacher', runs=len(v), values=list(np.round(v, 4)), mean=np.mean(v), sd=np.std(v, ddof=1) if len(v) > 1 else np.nan))
v = [json.load(open(SEEDS1 + 'results/teacher_sqrtweights_seed42.json'))['tests']['own']['macro_f1']] + \
    [own(result_path(f'teacher_sqrtweights_config1_merged_seed{s}')) for s in (1, 2) if os.path.exists(result_path(f'teacher_sqrtweights_config1_merged_seed{s}'))]
rows.append(dict(config='config1_merged', model='teacher_sqrtweights', runs=len(v), values=list(np.round(v, 4)), mean=np.mean(v), sd=np.std(v, ddof=1) if len(v) > 1 else np.nan))
df = pd.DataFrame(rows); print(df.round(4).to_string(index=False))
save_atomic_json(dict(table=json.loads(df.to_json(orient='records'))), OUT + 'summary.json')